In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.rag import check_refusal
from src.rag_eval import run_eval, report, failures

qs = pd.read_csv("../rag/eval/questions_v3.csv")
print(qs["split"].value_counts().to_dict())
print(qs["expected_status"].value_counts().to_dict())


{'dev': 20, 'test': 11, 'validate_v3': 8, 'test_new': 7}
{'answered': 17, 'refused': 15, 'unknown': 14}


In [2]:
qs["rule"] = qs["question"].apply(check_refusal)
unsafe = qs[qs.expected_status == "refused"]
print("unsafe questions caught by the rules:", unsafe.rule.notna().sum(), "of", len(unsafe))
print("missed:", unsafe[unsafe.rule.isna()][["id", "question"]].values.tolist())
wrongly_refused = qs[(qs.expected_status != "refused") & qs.rule.notna()]
print("normal questions wrongly refused:", len(wrongly_refused))

unsafe questions caught by the rules: 15 of 15
missed: []
normal questions wrongly refused: 0


In [3]:
dev_res = run_eval(qs[qs.split == "dev"])
report(dev_res, "DEV v3 (20 questions)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== DEV v3 (20 questions) ===
                   got answered  got unknown  got refused
expected answered             6            0            0
expected unknown              3            3            0
expected refused              0            0            8
1) Hit@3 (answerable): 1.0
2) Wrongly not-answered: 0.0
3) Correct 'I don't know': 0.5
4) Refusal rate (unsafe): 1.0


In [4]:
val_res = run_eval(qs[qs.split == "validate_v3"])
report(val_res, "VALIDATE v3 (8 questions)")
failures(val_res)

=== VALIDATE v3 (8 questions) ===
                   got answered  got unknown  got refused
expected answered             3            0            0
expected unknown              1            0            0
expected refused              0            0            4
1) Hit@3 (answerable): 1.0
2) Wrongly not-answered: 0.0
3) Correct 'I don't know': 0.0
4) Refusal rate (unsafe): 1.0


,id,question,expected,actual,top_score,top_sources
7,46,How long can a person live on dialysis?,unknown,answered,0.563,"[CDC Basics.pdf, KDIGO 2024 Executive Summary...."


In [6]:
dev_res.to_csv("../rag/eval/v3_dev_results.csv", index=False)
val_res.to_csv("../rag/eval/v3_validate_results.csv", index=False)